In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch

# Loading the data

In [41]:
try: # KAGGLE
    df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
except: 
    try: # COLAB
        df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
    except: # LOCAL
        df = pd.read_csv('../data/train.csv')
        test_df = pd.read_csv('../data/test.csv')
        
df.shape, test_df.shape

((2000, 8), (500, 7))

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Converting the dataset

In [4]:
from datasets import Dataset
label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})

In [15]:
# using roberta for now
from transformers import AutoModelForMultipleChoice, AutoTokenizer

model = AutoModelForMultipleChoice.from_pretrained('roberta-base', dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

model, tokenizer

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


(RobertaForMultipleChoice(
   (roberta): RobertaModel(
     (embeddings): RobertaEmbeddings(
       (word_embeddings): Embedding(50265, 768, padding_idx=1)
       (token_type_embeddings): Embedding(1, 768)
       (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
       (dropout): Dropout(p=0.1, inplace=False)
       (position_embeddings): Embedding(514, 768, padding_idx=1)
     )
     (encoder): RobertaEncoder(
       (layer): ModuleList(
         (0-11): 12 x RobertaLayer(
           (attention): RobertaAttention(
             (self): RobertaSelfAttention(
               (query): Linear(in_features=768, out_features=768, bias=True)
               (key): Linear(in_features=768, out_features=768, bias=True)
               (value): Linear(in_features=768, out_features=768, bias=True)
               (dropout): Dropout(p=0.1, inplace=False)
             )
             (output): RobertaSelfOutput(
               (dense): Linear(in_features=768, out_features=768, bias=True)


In [6]:
# dataset.shape[0] 
# first_sentences = [[question] * 5 for question in dataset['prompt']]
# first_sentences[0]

# import itertools

# nested = [[1, 2], [3, 4]]
# flat = list(itertools.chain.from_iterable(first_sentences))
# len(flat), len(first_sentences)

In [8]:
# # SAMPLE TEST
# import itertools

# data_slice = dataset.select(range(100))
# print(data_slice.info)
# options = ['A', 'B', 'C', 'D', 'E']

# first_sentences = [[question] * 5 for question in data_slice['prompt']]
# second_sentences = [[data_slice[option][i] for option in options] for i in range(100)]

# # flattening the list to put into tokenizer
# first_sentences = list(itertools.chain.from_iterable(first_sentences))
# second_sentences = list(itertools.chain.from_iterable(second_sentences))

# tokenized = tokenizer(first_sentences, second_sentences, return_tensors="pt", padding='max_length', truncation=True, max_length=256)
# tokenized

DatasetInfo(features={'id': Value('int64'), 'prompt': Value('string'), 'A': Value('string'), 'B': Value('string'), 'C': Value('string'), 'D': Value('string'), 'E': Value('string'), 'answer': Value('string')})


{'input_ids': tensor([[    0, 40285,     5,  ...,     1,     1,     1],
        [    0, 40285,     5,  ...,     1,     1,     1],
        [    0, 40285,     5,  ...,     1,     1,     1],
        ...,
        [    0,   495,  5906,  ...,     1,     1,     1],
        [    0,   495,  5906,  ...,     1,     1,     1],
        [    0,   495,  5906,  ...,     1,     1,     1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [14]:
# print(tokenized['input_ids'].shape)
# print(tokenized['attention_mask'].shape)

# # have to reshape it back in 100, 5, 256
# for key in tokenized:
#     tokenized[key] = tokenized[key].view(100, 5, 256)

# print(tokenized['input_ids'].shape)
# print(tokenized['attention_mask'].shape)

# mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
# tokenized['labels'] = torch.tensor(
#     [mapping[ans] for ans in data_slice['answer']], 
#     dtype=torch.float32
# )
# print(tokenized['labels'])

torch.Size([100, 5, 256])
torch.Size([100, 5, 256])


RuntimeError: shape '[100, 5, 256]' is invalid for input of size 100

In [35]:
def process_dataset(dataset):
    import itertools
    options = ['A', 'B', 'C', 'D', 'E']
    
    batch_size = 100
    first_sentences = [[question] * 5 for question in dataset['prompt']]
    second_sentences = [[dataset[option][i] for option in options] for i in range(batch_size)]

    # flattening the list to put into tokenizer
    first_sentences = list(itertools.chain.from_iterable(first_sentences))
    second_sentences = list(itertools.chain.from_iterable(second_sentences))

    output = tokenizer(first_sentences, second_sentences, return_tensors="pt", padding='max_length', truncation=True, max_length=256)
    ## FOR THIS PROBLEM THE MODEL NEEDS IN THIS SHAPE : BATCH_SIZE, OPTIONS, MAX_LENGTH
    # have to reshape it back in batch_size, options, max_length
    for key in output:
        output[key] = output[key].view(batch_size, 5, 256)

    # ALSO HAVE TO CREATE THE LABELS FOR USING TRAINER
    if 'answer' in dataset and dataset['answer'] is not None:
        mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        output['labels'] = torch.tensor(
            [mapping[ans] for ans in dataset['answer']], 
            dtype=torch.int
        ) 

    return output
    

In [28]:
# 'answer' in dataset.column_names

True

In [36]:
encoded_dataset = dataset.map(
    process_dataset, 
    batched=True,
    batch_size=100,
    remove_columns=dataset.column_names # removing the actual columns from the dataset
)
encoded_dataset.set_format(type="torch")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [37]:
encoded_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})

# Building the trainer

In [38]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./mcqa_results",
    # per_device_train_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    processing_class=tokenizer # another modification none asked for
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,3.196080
20,3.232566
30,3.220194
40,3.226735
50,3.220784
60,3.232563
70,3.206717
80,3.214927
90,3.189056
100,3.145080


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=2.505198834737142, metrics={'train_runtime': 703.1134, 'train_samples_per_second': 8.533, 'train_steps_per_second': 0.533, 'total_flos': 3946630394880000.0, 'train_loss': 2.505198834737142, 'epoch': 3.0})

# Create Predictions

In [43]:
test_dataset = Dataset.from_pandas(test_df)
encoded_test = test_dataset.map(
    process_dataset, 
    batched=True,
    batch_size=100,
    remove_columns=test_dataset.column_names # removing the actual columns from the dataset
)

encoded_test

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 500
})

In [49]:
predictions = trainer.predict(encoded_test)
logits = predictions.predictions
logits

array([[ 1.1516086, -2.6744566, -0.9345147, -1.6294099, -1.5100962],
       [-1.9608016,  3.780811 , -1.8948225, -1.8561435, -0.8859133],
       [-4.046527 ,  0.2520648, -4.9512615, -4.4517117, -4.63004  ],
       ...,
       [-4.228256 , -3.2416868, -3.1512535, -3.4641805, -4.064947 ],
       [-1.6832731,  3.4142284, -2.2119737, -4.554645 ,  4.115288 ],
       [-4.7243176, -4.420673 ,  4.137067 , -4.742876 , -4.715884 ]],
      shape=(500, 5), dtype=float32)

In [50]:
torch.topk(torch.tensor(logits), k=3, dim=1) # picking top 3 answers from the 5 logits so dim=1 as 5 is at dim 1

torch.return_types.topk(
values=tensor([[ 1.1516, -0.9345, -1.5101],
        [ 3.7808, -0.8859, -1.8561],
        [ 0.2521, -4.0465, -4.4517],
        ...,
        [-3.1513, -3.2417, -3.4642],
        [ 4.1153,  3.4142, -1.6833],
        [ 4.1371, -4.4207, -4.7159]]),
indices=tensor([[0, 2, 4],
        [1, 4, 3],
        [1, 0, 3],
        ...,
        [2, 1, 3],
        [4, 1, 0],
        [2, 1, 4]]))

In [54]:
top_3 = torch.topk(torch.tensor(logits), k=3, dim=1).indices.numpy()
id2label = {v: k for k, v in label_mapping.items()}

top_3_predictions = [
    ' '.join([id2label[idx] for idx in row])
    for row in top_3
]
top_3_predictions[:10] # works

['A C E',
 'B E D',
 'B A D',
 'E D C',
 'C A D',
 'D C E',
 'E D C',
 'B E C',
 'C D E',
 'B E C']

In [57]:
submission = pd.DataFrame({
    'Id': test_df['id'],
    'Prediction': top_3_predictions
})
submission.head(10)

,Id,Prediction
0,1,A C E
1,2,B E D
2,3,B A D
3,4,E D C
4,5,C A D
...,...,...
495,496,A C B
496,497,C B D
497,498,C B D
498,499,E B A


In [58]:
submission.to_csv('submission.csv', index=False)